In [16]:
%pip install databricks-langchain langchain_community langchain

dbutils.library.restartPython()

  Using cached databricks_vectorsearch-0.65-py3-none-any.whl.metadata (2.8 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached deprecation-2.1.0-py2.py3-none-any.whl.metadata (4.6 kB)
INFO: pip is looking at multiple versions of sse-starlette to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 44.9 MB/s eta 0:00:00
Using cached databricks_vectorsearch-0.65-py3-none-any.whl (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 140.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

Error: The "path" argument must be of type string. Received type undefined

In [17]:
from databricks_langchain import ChatDatabricks

LLM_ENDPOINT_NAME = "databricks-claude-3-7-sonnet"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

In [25]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# -------------------------------------------------------
# 1. LLM Setup

# llm = ChatDatabricks(
#     endpoint="databricks-meta-llama-3-3-70b-instruct",
#     max_tokens=800
# )

parser = StrOutputParser()


# -------------------------------------------------------
# 2. Fetch Code from Delta.  --- > replace with Vector search if needed.

def fetch_code(spark, business_unit, country):

    df = spark.table("main.sandesk4.git_index_raw")
    df = df.filter(df.business_unit == business_unit)
    print(df.count())

    if country:
        df = df.filter(df.filename.startswith(f"{country.lower()}_"))
    
    print(df.count())

    result = df.collect()

    if not result:
        return None, None

    return result[0].filename, result[0].code


# -------------------------------------------------------
# 3. Build LLM Reasoning Chain

def build_analysis_chain():

    def prompt_builder(inputs):
        return f"""
You are a senior C# backend engineer.

User Issue:
{inputs['issue']}

Business Unit:
{inputs['business_unit']}

File Being Inspected:
{inputs['filename']}

Code:
{inputs['code']}

Carefully analyze the code and explain:

- Is there any filter logic?
- Feature flag restriction?
- Config exclusion?
- Conditional return blocking data?

Explain clearly what is causing the issue.
"""

    chain = (
        RunnablePassthrough()
        | prompt_builder
        | llm
        | parser
    )

    return chain


# -------------------------------------------------------
# 4. Agent Orchestrator

def agent_process_query(spark, user_input):

    business_unit = user_input["business_unit"]
    filename = user_input["filename"]
    issue = user_input["issue"]

    print("\nUser Input:", user_input)

    # Step 1: Deterministic lookup
    filename, code = fetch_code(spark, business_unit, filename)

    if not code:
        print("No matching file found.")
        return

    print("\nInspecting File:", filename)

    # Step 2: LLM reasoning
    chain = build_analysis_chain()

    result = chain.invoke({
        "business_unit": business_unit,
        "issue": issue,
        "filename": filename,
        "code": code
    })

    print("\nLLM Root Cause Analysis:\n")
    print(result)


# -------------------------------------------------------
# 5. Test
# -------------------------------------------------------

user_input = {
    "business_unit": "LRS",
    "filename": "Japan",
    "issue": "Japan data is missing from dashboard"
}

agent_process_query(spark, user_input)

User Input: {'business_unit': 'LRS', 'filename': 'Japan', 'issue': 'Japan data is missing from dashboard'}
2
1

Inspecting File: japan_data_processor.cs

LLM Root Cause Analysis:

# Code Analysis for Japan Data Issue

After analyzing the `japan_data_processor.cs` file, I've identified multiple issues preventing Japan data from appearing in the dashboard.

## Issues Found

### 1. Region Configuration Exclusion
The `GlobalConfig.EnabledRegions` list only contains "US" and "EU", but does not include "JP" (Japan):
```csharp
public static List<string> EnabledRegions = new List<string> { "US", "EU" };
```
This means the first check in the `Process` method will fail for Japan data.

### 2. Feature Flag Restriction
The Japan feature flag is explicitly disabled:
```csharp
public static bool EnableJapan = false;
```
This is the second check that would block Japan data processing.

### 3. Conditional Logic in Process Method
The `Process` method contains three sequential checks, any of which will 

In [26]:
df = spark.table("main.sandesk4.git_index_raw")
display(df)

chunk_id,business_unit,filename,file_path,code
1,LRS,japan_data_processor.cs,LRS/japan_data_processor.cs,"public static class GlobalConfig { public static List EnabledRegions = new List { ""US"", ""EU"" }; } public static class FeatureFlags { public static bool EnableJapan = false; } public class JapanDataProcessor { public List Process(string regionCode) { if (!GlobalConfig.EnabledRegions.Contains(regionCode)) { return new List(); } if (!FeatureFlags.EnableJapan) { return new List(); } if (regionCode != ""JP"") { return new List(); } return LoadJapanData(); } }"
2,LRS,us_data_processor.cs,LRS/us_data_processor.cs,"public static class GlobalConfig { public static List EnabledRegions = new List { ""US"", ""EU"" }; } public static class FeatureFlags { public static bool EnableJapan = false; } public class USDataProcessor { public List Process(string regionCode) { if (!GlobalConfig.EnabledRegions.Contains(regionCode)) { return new List(); } if (regionCode != ""US"") { return new List(); } return LoadUSData(); } }"
3,Digital360,japan_sales.cs,Digital360/japan_sales.cs,"public static class GlobalConfig { public static List EnabledRegions = new List { ""US"", ""EU"" }; } public static class FeatureFlags { public static bool EnableJapan = false; } public class JapanSales { public int GetSales(string country) { if (!GlobalConfig.EnabledRegions.Contains(country)) { return 0; } if (country == ""JP"") { return 100; } return 0; } }"
